# 19.6 双重差分 / Difference-in-Differences (DiD)

**中文**：倾向匹配(19.5)要求"所有混杂都测到了",很苛刻。**双重差分(DiD)** 换了个巧妙思路:利用一个**自然实验**——某个政策/改动只作用于**一个群体**、在**某个时间点**发生。它对比"处理组的前后变化"与"对照组的前后变化",**两次相减**,一举消掉两类偏差:①群体本身的固定差异(处理组和对照组本来就不一样);②所有群体共有的时间趋势(经济大环境、季节)。本节用经济学史上最著名的自然实验——**Card-Krueger 最低工资研究**——演示,它还颠覆了教科书的经济学直觉。
**English**: Propensity matching (19.5) demands "all confounders are measured" — very strict. **Difference-in-Differences (DiD)** takes a clever route: exploit a **natural experiment** — a policy/change that hits **one group** at **one point in time**. It compares "the treated group's before→after change" with "the control group's before→after change," subtracting **twice** to cancel two kinds of bias at once: ① fixed group differences (treated and control differ to begin with); ② common time trends shared by all groups (the macro economy, seasonality). We demonstrate on economics' most famous natural experiment — the **Card-Krueger minimum-wage study** — which overturned textbook economic intuition.

---

**中文**：DiD 的估计量就是名字本身——**差分的差分**:
**English**: The DiD estimator is its name — the **difference of differences**:

$$\hat\tau_{\text{DiD}}=\underbrace{(\bar Y^{\text{treat}}_{\text{after}}-\bar Y^{\text{treat}}_{\text{before}})}_{\text{处理组的变化}}-\underbrace{(\bar Y^{\text{control}}_{\text{after}}-\bar Y^{\text{control}}_{\text{before}})}_{\text{对照组的变化}}$$

**中文**：直觉:处理组的前后变化里,既有**真实处理效应**,也有**时间趋势**(大家都在变)。对照组的前后变化里**只有时间趋势**(它没被处理)。两者一减,时间趋势被消掉,剩下的就是纯处理效应。**同时**,因为都是"各自的前后变化",群体本身的固定差异(基线高低)也被消掉了。**一次相减除掉时间趋势,另一次除掉群体差异——所以叫"双重"差分。**
**English**: Intuition: the treated group's before→after change contains both the **true effect** and the **time trend** (everything drifts). The control group's change contains **only the time trend** (it wasn't treated). Subtracting cancels the time trend, leaving the pure effect. **Simultaneously**, because both are "each group's own change," fixed group differences (baseline levels) cancel too. **One subtraction removes the time trend, the other removes the group difference — hence "double" differences.**

**中文**：等价地,可以用一个**带交互项的回归**估计(更方便加控制变量、算标准误):
**English**: Equivalently, estimate with a **regression with an interaction term** (convenient for adding controls and computing SEs):

$$Y = \beta_0 + \beta_1\,\text{treat} + \beta_2\,\text{post} + \beta_3\,(\text{treat}\times\text{post}) + \varepsilon$$

**中文**：交互项系数 $\beta_3$ 就是 DiD 效应(treat 吸收群体差异、post 吸收时间趋势、交互项捕捉"处理组在处理后额外多出来的那部分")。
**English**: The interaction coefficient $\beta_3$ is the DiD effect (treat absorbs the group difference, post absorbs the time trend, and the interaction captures "the extra part the treated group gains after treatment").

**中文**：DiD 的**命门是一个假设——平行趋势(parallel trends)**:*"如果没有处理,处理组和对照组本会沿着平行的轨迹变化。"* 只有这样,"用对照组的变化代表处理组本会有的变化"才成立。这个假设对处理后不可直接验证,但可以**检查处理前的趋势是否平行**来增强可信度。
**English**: DiD's crux is one assumption — **parallel trends**: *"absent treatment, the treated and control groups would have moved along parallel paths."* Only then does "using the control group's change to represent the treated group's counterfactual change" hold. Untestable directly for the post period, but you strengthen credibility by **checking whether pre-treatment trends were parallel**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 政策/自然实验必考）**
> **中文**：**DiD=(处理组前后变化)−(对照组前后变化)**, 一次消时间趋势、一次消群体固定差异。回归形式:Y~treat+post+**treat×post**, 交互项=效应。**核心假设=平行趋势**(无处理时两组走势平行)——**不可直接验证**, 用**事件研究(event study)** 画处理前各期的效应看是否≈0来佐证。优点:**允许存在未观测的、时间不变的混杂**(被组内差分消掉)——比 PSM 的假设弱。**坑**:①平行趋势被违反(如处理组本就在加速)→ 偏;②**处理效应异质 + 错峰采纳(staggered adoption)** 会让传统双向固定效应 DiD 有偏(2020后热点, 用 Callaway-Sant'Anna 等新估计量);③ SUTVA(溢出效应)。Card-Krueger:NJ 涨最低工资、PA 没涨→就业**没降反微升**, 颠覆"最低工资减少就业"的教科书直觉。
> **English**: **DiD = (treated before→after change) − (control before→after change)**, canceling the time trend once and the fixed group difference once. Regression: Y~treat+post+**treat×post**, the interaction = the effect. **Core assumption = parallel trends** (absent treatment the groups move in parallel) — **not directly testable**, supported by an **event study** (plot per-period pre-treatment effects, they should be ≈0). Advantage: **allows unobserved but time-invariant confounders** (differenced away within groups) — a weaker assumption than PSM. **Pitfalls**: ① violated parallel trends (e.g. the treated group was already accelerating) → bias; ② **heterogeneous effects + staggered adoption** bias classic two-way fixed-effects DiD (a hot post-2020 topic; use Callaway-Sant'Anna and similar new estimators); ③ SUTVA (spillovers). Card-Krueger: NJ raised the minimum wage, PA didn't → employment **did not fall, even rose slightly**, overturning the textbook "minimum wage reduces employment."


In [ ]:

# ============================================================
# 构造面板数据(已知真实效应)/ construct a panel with a KNOWN true effect
# 中文:两个群体(处理/对照), 多个时期。处理组和对照组基线不同(群体差异), 都有共同时间趋势,
#      处理组在 t0 之后多出真实效应 +3。我们看三种估计谁能还原这个 3。
# English: two groups (treated/control), many periods. Different baselines (group diff), a common time trend,
#      and the treated group gains a true effect +3 after t0. We see which estimator recovers the 3.
# ============================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
rng=np.random.default_rng(0)
true_effect=3.0; t0=5; periods=10; n_unit=200
rows=[]
for g in [0,1]:                                              # 0=对照, 1=处理 / control, treated
    baseline = 20 if g==1 else 30                            # 群体固定差异(处理组基线更低)/ group difference
    for u in range(n_unit):
        indiv = rng.normal(0,3)                              # 个体固定效应 / unit fixed effect
        for t in range(periods):
            time_trend = 1.5*t                               # 共同时间趋势 / common time trend
            post = int(t>=t0)
            y = baseline + indiv + time_trend + true_effect*(g==1)*post + rng.normal(0,2)
            rows.append((g,u+g*n_unit,t,post,y))
df=pd.DataFrame(rows,columns=["treat","id","t","post","y"])
print(f"面板:{df.id.nunique()} 个体 × {periods} 期; 处理在 t>={t0} 生效, 真实效应=+{true_effect}")


In [ ]:

# ============================================================
# 三种估计:两种朴素(都错) vs DiD / two naive (both wrong) vs DiD
# ============================================================
def cell(g,p): return df[(df.treat==g)&(df.post==p)].y.mean()
# ① 朴素前后比(只看处理组): 被时间趋势污染 / naive before-after (treated only): time-trend biased
naive_ba = cell(1,1)-cell(1,0)
# ② 朴素处理-对照比(只看处理后): 被群体差异污染 / naive treated-vs-control (after only): group-diff biased
naive_tc = cell(1,1)-cell(0,1)
# ③ DiD: 双重相减 / difference-in-differences
did = (cell(1,1)-cell(1,0)) - (cell(0,1)-cell(0,0))
# DiD 回归(带交互项, 附标准误)/ DiD regression with interaction (gives SE)
reg = smf.ols("y ~ treat*post", df).fit()

print(f"真实效应 / true effect: +{true_effect}")
print(f"① 朴素前后比(处理组)/ naive before-after: {naive_ba:+.2f}  (被 +时间趋势 污染, 高估)")
print(f"② 朴素处理-对照(处理后)/ naive treat-ctrl: {naive_tc:+.2f}  (被 群体基线差 污染, 低估甚至反号)")
print(f"③ 双重差分 DiD:            {did:+.2f}  ← 还原真值!")
print(f"③ DiD 回归交互项系数:      {reg.params['treat:post']:+.2f} (SE {reg.bse['treat:post']:.2f}, p={reg.pvalues['treat:post']:.1e})")


**中文**：两个朴素估计错得离谱且**方向相反**(一个因时间趋势高估、一个因群体差异低估),而 DiD 把两种偏差都消掉、精确还原了真值 +3。**DiD 的可信度全押在平行趋势上**——下面画出两组的完整轨迹:处理前它们应当平行(可检验),处理后处理组才因效应而"抬升"。
**English**: The two naive estimates are wildly wrong in **opposite directions** (one overestimates from the time trend, one underestimates from the group difference), while DiD cancels both and exactly recovers the truth +3. **DiD's credibility rides entirely on parallel trends** — below we plot both groups' full trajectories: before treatment they should be parallel (checkable), and only after does the treated group "lift" due to the effect.


In [ ]:

# ============================================================
# 可视化:平行趋势 + DiD 图 / parallel trends + the DiD diagram
# ============================================================
means=df.groupby(["treat","t"]).y.mean().unstack(0)         # 各组每期均值 / group means per period
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 平行趋势图 / trajectories
ax[0].plot(means.index, means[1], "o-", color="#C44E52", label="处理组 treated")
ax[0].plot(means.index, means[0], "o-", color="#4C72B0", label="对照组 control")
# 反事实:处理组若无处理会沿平行线走 / counterfactual: parallel to control after t0
cf = means[1][t0-1] + (means[0]-means[0][t0-1])
ax[0].plot(means.index[t0-1:], cf[t0-1:], "--", color="#C44E52", alpha=0.6, label="处理组反事实(平行)")
ax[0].axvline(t0-0.5, ls=":", color="gray", label="处理开始"); ax[0].axvspan(0,t0-0.5,alpha=0.05,color="green")
# 标出 DiD 效应(处理后真实 vs 反事实的差)/ mark the DiD gap
ax[0].annotate("", xy=(periods-1,means[1][periods-1]), xytext=(periods-1,cf[periods-1]),
               arrowprops=dict(arrowstyle="<->",color="green",lw=2))
ax[0].text(periods-1.5, (means[1][periods-1]+cf[periods-1])/2, f"DiD\n≈{did:.1f}", color="green", fontsize=9)
ax[0].set_title("平行趋势 + DiD:处理前平行(可查), 处理后抬升 / parallel trends + DiD gap")
ax[0].set_xlabel("时期 t"); ax[0].set_ylabel("结果 Y"); ax[0].legend(fontsize=8)
# ② 估计对比 / estimate comparison
ax[1].bar(["朴素前后\nbefore-after","朴素处理-对照\ntreat-ctrl","DiD","真值\ntruth"],
          [naive_ba,naive_tc,did,true_effect], color=["#DD8452","#C44E52","#55A868","#4C72B0"])
ax[1].axhline(true_effect,ls="--",color="#4C72B0"); ax[1].axhline(0,color="k",lw=0.5)
for i,v in enumerate([naive_ba,naive_tc,did,true_effect]): ax[1].text(i,v,f"{v:+.1f}",ha="center",va="bottom" if v>=0 else "top",fontsize=9)
ax[1].set_title("只有 DiD 还原真值 / only DiD recovers the truth"); ax[1].set_ylabel("估计效应")
plt.tight_layout(); plt.savefig("/tmp/ci06_viz.png",dpi=80); plt.show()
print("处理前两组轨迹平行(绿色区)→ 平行趋势假设可信 → DiD 估计可信")


**中文**：**真实案例:Card-Krueger 最低工资研究(1994)**。1992 年 4 月新泽西(NJ)把最低工资从 \$4.25 提到 \$5.05,而隔壁宾州(PA)没变。经济学教科书预测:最低工资上涨→企业雇不起人→就业下降。Card 和 Krueger 用 DiD 对比两州快餐店的就业变化,得到颠覆性结论:
**English**: **Real case: Card-Krueger minimum-wage study (1994)**. In April 1992 New Jersey (NJ) raised its minimum wage from \$4.25 to \$5.05, while neighboring Pennsylvania (PA) did not. Textbook economics predicts: higher minimum wage → firms can't afford workers → employment falls. Using DiD on fast-food employment across the two states, Card and Krueger reached an overturning conclusion:


In [ ]:

# ============================================================
# Card-Krueger 的经典 2x2 结果(论文汇总数据)/ Card-Krueger's classic 2x2 (paper summary)
# 中文:每家餐厅的全职等效就业人数(FTE)。NJ=处理组(涨了最低工资), PA=对照组。
# English: full-time-equivalent (FTE) employment per restaurant. NJ=treated (raised min wage), PA=control.
# ============================================================
ck = {"NJ (处理/treated)": {"before":20.44, "after":21.03},   # 来自 Card-Krueger 1994 表3 / from the paper
      "PA (对照/control)": {"before":23.33, "after":21.17}}
nj_change = ck["NJ (处理/treated)"]["after"] - ck["NJ (处理/treated)"]["before"]
pa_change = ck["PA (对照/control)"]["after"] - ck["PA (对照/control)"]["before"]
ck_did = nj_change - pa_change
print("Card-Krueger 快餐店全职就业(FTE)/ fast-food FTE employment:")
for s,v in ck.items(): print(f"  {s}: 涨薪前 {v['before']:.2f} → 涨薪后 {v['after']:.2f}  (变化 {v['after']-v['before']:+.2f})")
print(f"\nNJ 变化 {nj_change:+.2f}, PA 变化 {pa_change:+.2f}")
print(f"DiD = {nj_change:+.2f} − ({pa_change:+.2f}) = {ck_did:+.2f} FTE")
print("→ 涨最低工资后, NJ 就业相对 PA 不降反升! 颠覆'最低工资减少就业'的教科书直觉")
print("→ raising the minimum wage did NOT reduce employment (even rose vs PA) — a landmark finding")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **DiD 用"双重相减"消掉两类偏差**:朴素前后比被时间趋势毁掉(高估到 +10),朴素处理-对照比被群体基线差毁掉(甚至反号),而 DiD 同时消掉这两者、精确还原 +3。它的强大之处在于**允许存在未观测的、不随时间变的混杂**(如"处理组的地区文化")——这些会被组内前后差分自动消掉,所以 DiD 的假设比倾向匹配**更弱、更可信**。
2. **但一切都押在平行趋势上**:DiD 唯一(也是最脆弱)的假设是"无处理时两组会平行变化"。如果处理组在处理前就已经在**加速/减速**(比如 NJ 经济本来就更热),DiD 就会把这个差异误当成处理效应。**处理后无法验证,但一定要画处理前的趋势(事件研究图)看是否平行**——这是审稿人和面试官必问的。
3. **Card-Krueger 的启示与争议**:它用一个干净的自然实验,拿数据挑战了"最低工资必减就业"的教条,是**实证经济学(credibility revolution)** 的里程碑(Card 因此获 2021 诺奖)。但也要诚实:①后续有学者用不同数据/方法得到不同结论(因果推断的结果对方法敏感);②**近年发现**:当处理在不同时间**错峰**发生、且效应异质时,传统双向固定效应 DiD 会有偏——这是 2020 年后计量经济学的热点,催生了 Callaway-Sant'Anna、Sun-Abraham 等新估计量。**别把 DiD 当万能药,平行趋势和错峰采纳是它的两大软肋。**

**English**:
1. **DiD cancels two biases via "double subtraction"**: the naive before-after is ruined by the time trend (overestimating to +10), the naive treated-vs-control is ruined by the group baseline difference (even wrong-signed), while DiD cancels both and exactly recovers +3. Its strength is **allowing unobserved, time-invariant confounders** (like "the treated region's culture") — these are automatically differenced away within groups, so DiD's assumption is **weaker and more credible** than propensity matching.
2. **But everything rides on parallel trends**: DiD's sole (and most fragile) assumption is "absent treatment the groups would move in parallel." If the treated group was already **accelerating/decelerating** before treatment (say NJ's economy was already hotter), DiD would mistake that for the treatment effect. **Untestable post-treatment, but you must plot pre-treatment trends (an event-study plot) to check parallelism** — reviewers and interviewers always ask.
3. **Card-Krueger's lesson and controversy**: with a clean natural experiment, it used data to challenge the dogma that "minimum wage must cut employment," a milestone of the **credibility revolution** in empirical economics (Card won the 2021 Nobel). But honestly: ① later scholars using different data/methods reached different conclusions (causal results are method-sensitive); ② **recently discovered**: when treatment occurs at **staggered** times with heterogeneous effects, classic two-way fixed-effects DiD is biased — a hot post-2020 econometrics topic that spawned new estimators (Callaway-Sant'Anna, Sun-Abraham). **Don't treat DiD as a cure-all; parallel trends and staggered adoption are its two weaknesses.**

> 💼 **实战视角 / Practical angle**
> **中文**:DiD 是**政策评估与产品分析的主力**(某地上线新功能、某国改了规则、分批 rollout 时用它估效果)。落地要点:①**先画事件研究图**(处理前各期效应≈0=平行趋势成立), 这是最强的可信度证据;②回归里加**个体和时间固定效应**(双向固定效应)+ 聚类稳健标准误(按处理单元聚类);③**错峰 rollout** 慎用传统 DiD, 用新估计量;④找**尽可能相似**的对照组;⑤配合安慰剂检验(假装处理提前发生, 应无效应)。面试金句:*"DiD 用(处理组前后变化)−(对照组前后变化), 一次消时间趋势一次消群体固定差异, 允许时间不变的未观测混杂; 命门是平行趋势——用事件研究图查处理前趋势; 错峰采纳+异质效应会让传统 DiD 有偏。"*
> **English**: DiD is a **workhorse of policy evaluation and product analysis** (a feature launched in one region, a country changing rules, a staged rollout). Deployment keys: ① **plot the event study first** (pre-treatment per-period effects ≈0 = parallel trends), the strongest credibility evidence; ② add **unit and time fixed effects** (two-way FE) + cluster-robust SEs (cluster by treatment unit); ③ for **staggered rollouts** avoid classic DiD, use the new estimators; ④ pick the **most similar** control group; ⑤ pair with placebo tests (pretend treatment happened earlier — should show no effect). Interview line: *"DiD is (treated change) − (control change), canceling the time trend once and the fixed group difference once, allowing time-invariant unobserved confounders; the crux is parallel trends — check pre-trends with an event study; staggered adoption + heterogeneous effects bias classic DiD."*

---
### 小结 / Summary
- **中文**:DiD=(处理组前后变化)−(对照组前后变化), 同时消时间趋势与群体固定差异; 回归用 treat×post 交互项。
- **English**: DiD = (treated change) − (control change), canceling both the time trend and the fixed group difference; regression uses the treat×post interaction.
- **中文**:核心假设=平行趋势(不可直接验证, 用事件研究查处理前趋势); 允许时间不变的未观测混杂。
- **English**: Core assumption = parallel trends (untestable directly, check pre-trends via event study); allows time-invariant unobserved confounders.
- **中文**:Card-Krueger:最低工资上涨未减就业, 实证经济学里程碑; 注意错峰采纳偏差。
- **English**: Card-Krueger: a minimum-wage rise didn't cut employment, a milestone of empirical economics; mind staggered-adoption bias.
